<img src='../images/magic.png'>

### Magic: The Gathering - Data Wrangling
* Physical Cards, English Only, Secondary Market Price - Retail

In [1]:
import pandas as pd

#### Clean the card data.

In [2]:
# 'low_memory = False' ensures consistent data types by reading the whole column instead of in chunks which results in "mixed" data types.
dfm = pd.read_csv('../data/dataMagic/cardsMagic.csv', low_memory = False)

# Specify the needed columns.
dfm = dfm[["availability", "colors", "language", "name", "rarity", "setCode", "types", "uuid"]]

# Replace all NaN from source data with "C" for "Colorless".
dfm["colors"] = dfm["colors"].fillna("C")

# Specifying the rows to keep involving paper.
dfm = dfm[
    (dfm["availability"] == "mtgo, paper") | 
    (dfm["availability"] == "paper") | 
    (dfm["availability"] == "arena, mtgo, paper") |
    (dfm["availability"] == "arena, paper") 
    ]

# Keep only the English card versions.
dfm = dfm[dfm["language"] == "English"]

# Remove the basic lands from each set to tighten the dataframe.  
# These lands are printed for most sets in bulk and are mostly worthless, barring outliers.  
basic_lands = ["Forest", "Island", "Mountain", "Plains", "Swamp"]
dfm = dfm[~dfm["name"].isin(basic_lands)]

#### Add and merge sets CSV.

In [3]:
dfmSets = pd.read_csv('../data/dataMagic/setsMagic.csv')

In [4]:
dfm2 = pd.merge(dfm, dfmSets, on = "setCode", how = "inner")

#### Add and merge prices CSV.

In [5]:
dfmPrices =  pd.read_csv('../data/dataMagic/pricesMagic.csv')

# Rename so there is no confusion with releaseDate.
dfmPrices = dfmPrices.rename(columns = {"date" : "sourceDate"})

# Convert source date to datetime.
dfmPrices['sourceDate'] = pd.to_datetime(dfmPrices['sourceDate'], format = '%Y-%m-%d', errors = 'raise')

# Specify and remove the types we do not want in the dataframe:
# MTGO is Magic: The Gathering Online which is digital poduct.
# Buylist is what stores will pay for cards, and we only want retail prices.
# Cardmarket is an online retailer located in Germany, thus their prices are in EUR.
mtgo = ["mtgo"]
buylist = ["buylist"]
cardmarket = ["cardmarket"]
dfmPrices = dfmPrices[~dfmPrices["gameAvailability"].isin(mtgo)]
dfmPrices = dfmPrices[~dfmPrices["providerListing"].isin(buylist)]
dfmPrices = dfmPrices[~dfmPrices["priceProvider"].isin(cardmarket)]

In [6]:
dfm3 = pd.merge(dfm2, dfmPrices, on = "uuid", how = "left")

#### dfm3 will be used for SQL queries and individual price lookups.

In [7]:
# Add a colum for average market price.
dfm3["avgMarketPrice"] = dfm3.groupby(['uuid', 'cardFinish'])['price'].transform('mean')

# Round to 2 decimal places.
dfm3["avgMarketPrice"] = dfm3["avgMarketPrice"].round(2)

# Availability is no longer needed, since we now have gameAvailability from pricesMagic.csv.
dfm3.drop(columns = ["availability"], inplace = True)

# Convert release dates to datetime for plotting.
dfm3['releaseDate'] = pd.to_datetime(dfm3['releaseDate'], format = '%m/%d/%Y', errors = 'raise')

# Make a more viewer-friendly column and sort order.
newOrderM = ['name', 'setCode', 'setName', 'language', 'types', 'colors', 'rarity', 'cardFinish', 'releaseDate', 'releaseYear', 'gameAvailability', 
             'priceProvider', 'price', 'avgMarketPrice', 'currency', 'providerListing', 'sourceDate', 'uuid']
dfm3 = dfm3[newOrderM]

# Sort by release date, then set name, then card name
dfm3 = dfm3.sort_values(by=["releaseDate", "setName", "name"])

# Standardize capitalization for these 3 columns.
dfm3['rarity'] = dfm3['rarity'].str.title()
dfm3['cardFinish'] = dfm3['cardFinish'].str.title()
dfm3['gameAvailability'] = dfm3['gameAvailability'].str.title()

# Reset index after manipulation and to check new number of rows.
# Drop the original index column.
dfm3 = dfm3.reset_index(drop = True)

In [8]:
# Save for SQL reference.
dfm3.to_csv("../data/dataMagic/cleanMagicIndPrices.csv", index = False)

# Make a pickle file if CSV file size is too large.
# dfm3.to_pickle("../dataMagic/cleanMagicIndPrices.pkl")

#### dfm4 will be used for visualization and as a final, cleaner version.

In [9]:
# Individual prices and price providers not relevant for data viz.
dfm4 = dfm3.drop(columns = ["price", "priceProvider"])

# Remove dupes.
dfm4.drop_duplicates(keep = "first", inplace = True)

# Reset index again.
dfm4 = dfm4.reset_index(drop = True)

In [10]:
# Save for visualization reference and to serve as final dataframe.
dfm4.to_csv("../data/dataMagic/cleanMagic.csv", index = False)

# Make a pickle file if CSV file size is too large.
# dfm4.to_pickle("../data/dataMagic/cleanMagic.pkl")

### Search check to ensure functionality.

In [11]:
dfm3[dfm3["name"] == "Presence of the Master"]

,name,setCode,setName,language,types,colors,rarity,cardFinish,releaseDate,releaseYear,gameAvailability,priceProvider,price,avgMarketPrice,currency,providerListing,sourceDate,uuid
7586,Presence of the Master,LEG,Legends,English,Enchantment,W,Uncommon,Normal,1994-06-01,1994,Paper,tcgplayer,10.53,10.03,USD,retail,2025-12-08,3c540848-d4ca-5441-a098-51a295e39aef
7587,Presence of the Master,LEG,Legends,English,Enchantment,W,Uncommon,Normal,1994-06-01,1994,Paper,cardkingdom,9.99,10.03,USD,retail,2025-12-08,3c540848-d4ca-5441-a098-51a295e39aef
7588,Presence of the Master,LEG,Legends,English,Enchantment,W,Uncommon,Normal,1994-06-01,1994,Paper,cardsphere,10.52,10.03,USD,retail,2025-12-08,3c540848-d4ca-5441-a098-51a295e39aef
7589,Presence of the Master,LEG,Legends,English,Enchantment,W,Uncommon,Normal,1994-06-01,1994,Paper,manapool,9.09,10.03,USD,retail,2025-12-08,3c540848-d4ca-5441-a098-51a295e39aef
25339,Presence of the Master,USG,Urza's Saga,English,Enchantment,W,Uncommon,Normal,1998-10-12,1998,Paper,tcgplayer,0.43,0.44,USD,retail,2025-12-08,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
25340,Presence of the Master,USG,Urza's Saga,English,Enchantment,W,Uncommon,Normal,1998-10-12,1998,Paper,cardkingdom,0.59,0.44,USD,retail,2025-12-08,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
25341,Presence of the Master,USG,Urza's Saga,English,Enchantment,W,Uncommon,Normal,1998-10-12,1998,Paper,cardsphere,0.44,0.44,USD,retail,2025-12-08,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
25342,Presence of the Master,USG,Urza's Saga,English,Enchantment,W,Uncommon,Normal,1998-10-12,1998,Paper,manapool,0.30,0.44,USD,retail,2025-12-08,cd9ed8e9-3778-5e5c-907e-db5f41dbc215
